# 25 — Prompt Governance and Responsible AI

    ## Scenario and success criteria

    A high-risk support artifact may ship only with an active, policy-versioned approval from the trusted governance registry.

    This guided lab succeeds when its assertions pass and the learner can explain why the baseline fails, what the mitigation changes, and which production controls remain outside the simulation.

    ## Learning objectives

    - Connect policy to executable controls.
- Keep ownership and prohibited uses explicit.
- Use trusted approval records rather than model claims.

    **Prerequisites:** Courses 01–13 and the preceding advanced/enterprise lesson.
    **Safety boundary:** all behavior is deterministic and synthetic; there are no credentials, external calls, or side effects. Printed results are simulation evidence, not a live-model benchmark.

## Mental model and architecture

![Course 25 architecture](diagram-1.svg)

Treat the model as one uncertain component inside a deterministic control plane. Inputs, schemas, identity, authorization, metrics, release gates, and state transitions remain application responsibilities.

## Baseline and failure injection

A mutable boolean inside a manifest or model output is not auditable approval.

The next cell defines the synthetic fixture and the smallest reusable primitive needed to make that failure observable.

In [ ]:
from lab25 import Approval, GovernanceManifest, governance_gate

manifest = GovernanceManifest(
    artifact_id="support-finance-v2", artifact_digest="aaaaaaaaaaaaaaaa", owner="support-ai", risk_tier="high",
    handles_personal_data=True, intended_use="explain billing records", prohibited_uses=("credit decision",),
)
NOW = 2_000
valid = Approval("APR-1", "support-finance-v2", manifest.artifact_digest, "reviewer-7", "risk_board", "policy-2026-2", 3_000, "active")
wrong_version = Approval("APR-2", "support-finance-v2", manifest.artifact_digest, "reviewer-7", "risk_board", "policy-2025-4", 3_000, "active")

## Experiment

Run the baseline and candidate on the same fixture so the comparison is attributable.

In [ ]:
print("no approval", governance_gate(manifest, [], policy_version="policy-2026-2", now=NOW))
print("stale approval", governance_gate(manifest, [wrong_version], policy_version="policy-2026-2", now=NOW))
print("valid approval", governance_gate(manifest, [valid], policy_version="policy-2026-2", now=NOW))

## Evaluation

The assertions below are the executable contract. They validate both a positive path and a boundary or failure path; a printed claim alone is not proof.

In [ ]:
assert not governance_gate(manifest, [], policy_version="policy-2026-2", now=NOW).allowed
assert not governance_gate(manifest, [wrong_version], policy_version="policy-2026-2", now=NOW).allowed
assert governance_gate(manifest, [valid], policy_version="policy-2026-2", now=NOW).allowed

## Production upgrade

Maintain policy/control/evidence traceability, named owners, exception expiry, audit retention, incident escalation, and jurisdiction-specific review. Reassess when use, data, model, or policy changes.

| Teaching lab | Production system |
| --- | --- |
| Synthetic fixtures | Versioned, reviewed, privacy-safe datasets |
| Deterministic simulation | Provider adapter plus optional recorded replay |
| In-process state | Durable state with tenant and retention boundaries |
| Assertions | CI gates, staged rollout, monitoring, and rollback |

## Exercises

1. Add one normal, one boundary, and one adversarial case without weakening an invariant.
2. Change one design variable and report the metric numerator, denominator, unit, and direction.
3. Write a production decision memo that identifies owner, failure policy, monitoring signal, and rollback trigger.

## Takeaway

Use probabilistic components for bounded interpretation; use trusted deterministic code for permissions, validation, metrics, and consequential state changes.